In [32]:
import importlib

import numpy as np
import pandas as pd

import model_functions
import utils

importlib.reload(model_functions)
importlib.reload(utils)

from model_functions import load_transition_matrix, gen_risk_matrix, gen_transition_probabilities, run_markov_cohort, \
    run_mc_sim, calculate_outcomes, plot_trace, load_costs, STAGE_ORDER, run_comparison

# Validation
Running validation checks on the baseline transition probabilities to determine which data source to use for our CEA model. Will run on the observation-based and trial-based probabilities (as well as the low/high CI values for each).

In [6]:
suffix_defs = ['obs', 'trial', 'obs_low', 'obs_high', 'trial_low', 'trial_high']

## Average time to Decompensation
- F0/1 -> DC : 30-35 years
- F2 -> DC : 19 years
- F3 -> DC : 6 years
- F4 -> DC : 5.6 years

In [ ]:

# TODO - change prevalence to use
tm_stage_dc = load_transition_matrix('~/Documents/courses/HRP_392/final_baseline_transition_probs.csv', "_obs",
                                     cycle_length=1)
mc_trace = run_mc_sim(tm_stage_dc, [0.5, 0.5, 0, 0, 0, 0, 0, 0, 0, 0], n_cycles, 100, starting_age=starting_age,
                      cycle_length=1, risk_matrix=None)
liver = mc_trace[((mc_trace > 4) & (mc_trace < 9)).sum(axis=1).gt(0)]

# TODO check
((liver > 4).idxmax(axis=1).apply(lambda x: x[0]) - (liver > 1).idxmax(axis=1).apply(lambda x: x[0])).mean()

# Add other stage starts (or could do really big cohort and look at all the differences

## ESSENCE Trial Liver Fibrosis Improvement
- Initial prevalance: 5/16 F2, 11/16 F3
- Placebo N=266, 50 had improvement in Liver Fibrosis
- Semaglutide N=534, 183 had improvement in Liver Fibrosis



In [ ]:
essence_validation_results = []
n_iter = 100
for suf in suffix_defs:
    for i in range(n_iter):
        tm_mod = load_transition_matrix('~/Documents/courses/HRP_392/final_baseline_transition_probs.csv', f"_{suf}",
                                        cycle_length=12 / 52.14)
        cohort_size = 266
        mc_trace = run_mc_sim(tm_mod, [0, 5 / 16, 11 / 16, 0, 0, 0, 0, 0, 0, 0], 6, cohort_size, starting_age=56,
                              cycle_length=12 / 52.14)
        
        # Number of patients with fibrosis regression after 72 weeks
        stage_changes = (mc_trace.iloc[:, -1] - mc_trace.iloc[:, 0]).value_counts().sort_index()
        regression_count = stage_changes[stage_changes.index < 0].sum()
    
        essence_validation_results.append([suf, i, regression_count])

essence_val_df = pd.DataFrame(essence_validation_results)

In [70]:
essence_val_df.groupby(0)[2].describe()

,count,mean,std,min,25%,50%,75%,max
0,,,,,,,,
obs,2.0,12.5,6.363961,8.0,10.25,12.5,14.75,17.0
obs_high,2.0,17.5,0.707107,17.0,17.25,17.5,17.75,18.0
obs_low,2.0,6.5,3.535534,4.0,5.25,6.5,7.75,9.0
trial,2.0,46.0,2.828427,44.0,45.00,46.0,47.00,48.0
trial_high,2.0,57.0,8.485281,51.0,54.00,57.0,60.00,63.0
trial_low,2.0,38.0,8.485281,32.0,35.00,38.0,41.00,44.0


## Newsome Trial Stage changes

- Initial Prevalance for Placebo group: 22 F1, 22 F2, 36 F3 (missing 5, 3, and 2 pts in final data)


In [44]:
comparison = pd.DataFrame([[5, 2, 1],
                           [6, 4, 7],
                           [4, 6, 6],
                           [2, 6, 18],
                           [0, 1, 2]], index=[0, 1, 2, 3, 4], columns=[1, 2, 3])
comparison_props = comparison / comparison.sum(axis=0)

In [ ]:
# TODO - should I make the initial prevalence match the trial exactly?

In [64]:
newsome_validation_results = {}
n_iter = 10
for suf in suffix_defs:
    newsome_validation_results[suf] = np.zeros((5,3), dtype=object)
    suf_counts = np.zeros((n_iter, 10, 3))
    for i in range(n_iter):
        tm_mod = load_transition_matrix('~/Documents/courses/HRP_392/final_baseline_transition_probs.csv', f"_{suf}",
                                        cycle_length=12 / 52.14)
        cohort_size = 80
        mc_trace = run_mc_sim(tm_mod, [0, 22 / 80, 22 / 80, 36 / 80, 0, 0, 0, 0, 0, 0], 6, cohort_size, starting_age=55,
                              cycle_length=12 / 52.14)
        
        # Number of patients with fibrosis regression after 72 weeks
        stage_changes = (mc_trace.iloc[:, -1] - mc_trace.iloc[:, 0]).value_counts().sort_index()
        regression_count = stage_changes[stage_changes.index < 0].sum()
    
        counts = mc_trace.iloc[:, [0, -1]].value_counts().unstack().T.fillna(0)
        
        for j in range(10):
            if j not in counts.index:
                counts.loc[j, :] = [0, 0, 0]
        
        counts.sort_index(inplace=True)
        
        suf_counts[i,:,:] = counts
        
    suf_props = suf_counts / suf_counts.sum(axis=1, keepdims=True)
    
    for i in range(5):
        for j in range(3):
            lt = int((suf_props[:, i, j] < comparison_props.iloc[i, j]).sum())
            eq = int((suf_props[:, i, j] == comparison_props.iloc[i, j]).sum())
            gt = int((suf_props[:, i, j] > comparison_props.iloc[i, j]).sum())
            newsome_validation_results[suf][i, j] = (lt, eq, gt)



In [65]:
newsome_validation_results['trial']

array([[(10, 0), (10, 0), (9, 0)],
       [(0, 10), (9, 1), (10, 0)],
       [(6, 3), (0, 10), (5, 4)],
       [(10, 0), (9, 1), (1, 9)],
       [(0, 0), (10, 0), (3, 6)]], dtype=object)

In [67]:
newsome_validation_results['obs']

array([[(10, 0), (10, 0), (10, 0)],
       [(0, 10), (10, 0), (10, 0)],
       [(9, 1), (0, 10), (9, 1)],
       [(10, 0), (10, 0), (0, 10)],
       [(0, 0), (9, 1), (7, 3)]], dtype=object)

## Liver Transplant rates